# NLP Engine Benchmarking for Firespeaker Studio

This notebook evaluates different NLP engines (spaCy, NLTK, Azure) for tasks related to audiobook text analysis:
- Dialogue detection
- Character identification (NER)
- Emotion/sentiment analysis
- Processing speed for long-form content

In [1]:
# Import required libraries
import spacy
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import json
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime
import seaborn as sns
from pathlib import Path
import psutil
import gc  # For garbage collection
from tqdm.notebook import tqdm  # For progress bars

# On Windows, use spawn method instead of fork for multiprocessing compatibility
import multiprocessing
if os.name == 'nt':
    multiprocessing.set_start_method('spawn', force=True)
else:
    try:
        multiprocessing.set_start_method('fork', force=True)
    except RuntimeError:
        pass

# Download only the necessary NLTK resources (much faster than 'all')
nltk.download('punkt')  # For tokenization
nltk.download('maxent_ne_chunker')  # For NER
nltk.download('words')  # Required for NER
nltk.download('vader_lexicon')  # For sentiment analysis
nltk.download('averaged_perceptron_tagger')  # For POS tagging

def clear_memory():
    """Force garbage collection and print memory usage stats"""
    # Force garbage collection
    gc.collect()
    
    # Get memory info
    process = psutil.Process(os.getpid())
    memory_info = process.memory_info()
    
    print(f"Memory cleanup completed")
    print(f"Current memory usage: {memory_info.rss / (1024 * 1024):.2f} MB")
    print(f"Available system memory: {psutil.virtual_memory().available / (1024 * 1024 * 1024):.2f} GB")
    
    # Brief pause to allow OS to reclaim memory
    time.sleep(1)

# Clear memory before starting
print("Starting fresh run - cleaning up memory...")
clear_memory()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\TimmothyEscolopio\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\TimmothyEscolopio\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\TimmothyEscolopio\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\TimmothyEscolopio\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\TimmothyEscolopio\AppData\Roaming\nltk_data..
[nltk_data]     .
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       

Starting fresh run - cleaning up memory...
Memory cleanup completed
Current memory usage: 195.44 MB
Available system memory: 27.14 GB


In [2]:

# Set optimal parameters for your hardware
# With 64GB RAM and i7-11800H (8 cores/16 threads), we can be more aggressive
CHUNK_SIZE = 75000    # Larger chunks since you have plenty of RAM
OVERLAP = 1000        # Larger overlap for better context preservation
MAX_WORKERS = 6       # More workers, but still leaving headroom for system

# Run benchmarks on all samples using ProcessPoolExecutor with tqdm progress bar
all_results = []

In [37]:
# Load spaCy models
try:
    nlp_sm = spacy.load("en_core_web_sm")
    print("Loaded spaCy small model")
except OSError:
    print("Small model not found. Please run: python -m spacy download en_core_web_sm")
    
try:
    nlp_lg = spacy.load("en_core_web_lg")
    # Add spacytextblob to the pipeline if not already added
    if 'spacytextblob' not in nlp_lg.pipe_names:
        from spacytextblob.spacytextblob import SpacyTextBlob
        nlp_lg.add_pipe("spacytextblob")
    print("Loaded spaCy large model with SpacyTextBlob")
except OSError:
    print("Large model not found. Please run: python -m spacy download en_core_web_lg")
    print("Using small model as fallback")
    nlp_lg = nlp_sm
    if 'spacytextblob' not in nlp_lg.pipe_names:
        from spacytextblob.spacytextblob import SpacyTextBlob
        nlp_lg.add_pipe("spacytextblob")

Loaded spaCy small model
Loaded spaCy large model with SpacyTextBlob


In [38]:
# Load NLTK resources
try:
    # Check if resources are available by attempting to use them
    nltk.sent_tokenize("This is a test sentence.")
    nltk.word_tokenize("This is a test sentence.")
    nltk.pos_tag(["This", "is", "a", "test"])
    nltk.ne_chunk(nltk.pos_tag(nltk.word_tokenize("John Smith works at Google.")))
    SentimentIntensityAnalyzer().polarity_scores("This is a test.")
    print("NLTK resources loaded successfully")
except Exception as e:
    print(f"Error loading NLTK resources: {e}")
    print("Please uncomment and run the nltk.download() commands at the top of the notebook")

NLTK resources loaded successfully


In [39]:
# Create directories for data
os.makedirs("../data/corpus/classics", exist_ok=True)
os.makedirs("../data/corpus/modern", exist_ok=True)
os.makedirs("../data/corpus/dialogue_heavy", exist_ok=True)
os.makedirs("../data/corpus/descriptive", exist_ok=True)
os.makedirs("../data/corpus/emotional", exist_ok=True)
os.makedirs("../data/ground_truth", exist_ok=True)
os.makedirs("../results", exist_ok=True)

print("Directory structure created.")

Directory structure created.


In [28]:
# Uncomment and fill in your Azure credentials if you want to use Azure
'''
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

key = "your-key-here"
endpoint = "your-endpoint-here"

def get_azure_client():
    credential = AzureKeyCredential(key)
    client = TextAnalyticsClient(endpoint=endpoint, credential=credential)
    return client

azure_client = get_azure_client()
'''

# Set this to True if you want to include Azure in benchmarks
include_azure = False

## Create Project Directory Structure

## 1. Setup Data Loading Functions

In [41]:
def load_samples(directory="../data/corpus"):
    """Load sample texts from corpus directory, printing file loading information."""
    samples = {}
    print("Loading text samples from directory:", directory)  # Added for clarity
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(".txt"):
                file_path = os.path.join(root, filename)
                category = os.path.basename(os.path.dirname(file_path))
                sample_id = f"{category}/{filename}"
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        samples[sample_id] = f.read()
                        print(f"Loaded: {sample_id}")  # Print each file as it's loaded
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    return samples



def create_ground_truth(sample_id, text, dialogues=None, characters=None, emotions=None):
    """Create or update ground truth annotations for a text sample"""
    ground_truth = {
        "sample_id": sample_id,
        "text": text,
        "dialogues": dialogues or [],
        "characters": characters or [],
        "emotions": emotions or []
    }
    
    # Save to JSON
    os.makedirs("../data/ground_truth", exist_ok=True)
    with open(f"../data/ground_truth/{sample_id.replace('/', '_')}.json", "w") as f:
        json.dump(ground_truth, f, indent=2)
    
    return ground_truth

def load_ground_truth(sample_id):
    """Load ground truth annotations for a text sample"""
    path = f"../data/ground_truth/{sample_id.replace('/', '_')}.json"
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return None

## 2. Download Sample Texts (if needed)

In [ ]:
# Example function to download sample texts from Project Gutenberg
def download_gutenberg_sample(url, filename, category="classics"):
    """Download a sample text from Project Gutenberg"""
    import requests
    
    response = requests.get(url)
    if response.status_code == 200:
        filepath = f"../data/corpus/{category}/{filename}"
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"Downloaded {filename} to {filepath}")
        return True
    else:
        print(f"Failed to download {filename}. Status code: {response.status_code}")
        return False

# Example usage:
# Download Pride and Prejudice by Jane Austen
# download_gutenberg_sample(
#     "https://www.gutenberg.org/files/1342/1342-0.txt", 
#     "pride_and_prejudice.txt", 
#     "classics"
# )

## 3. Benchmark Implementation - Core NLP Tasks

In [42]:
# Dialogue Detection Functions

def dialogue_detection_spacy(text):
    """Detect dialogue using spaCy"""
    start_time = time.time()
    doc = nlp_lg(text)
    
    # Dialogue detection using quotation marks and speech verbs
    dialogue_sentences = []
    speech_verbs = ["say", "tell", "ask", "reply", "respond", "shout", "whisper", "mutter", 
                   "exclaim", "call", "cry", "yell", "speak", "answer", "murmur"]
    
    for sent in doc.sents:
        text = sent.text.strip()
        # Look for quotation marks
        has_quotes = ('"' in text) or ("'" in text)
        # Look for speech verbs
        has_speech_verb = any(token.lemma_ in speech_verbs for token in sent if token.pos_ == "VERB")
        
        if has_quotes and has_speech_verb:
            dialogue_sentences.append(text)
        # Also detect standalone quotes that might be dialogue
        elif has_quotes and (text.startswith('"') or text.startswith("'")):
            dialogue_sentences.append(text)
    
    end_time = time.time()
    return {
        "detected_dialogue": dialogue_sentences,
        "time_taken": end_time - start_time,
        "count": len(dialogue_sentences)
    }

def dialogue_detection_nltk(text):
    """Detect dialogue using NLTK"""
    start_time = time.time()
    
    sentences = nltk.sent_tokenize(text)
    dialogue_sentences = []
    speech_verbs = ["say", "tell", "ask", "reply", "respond", "shout", "whisper", "mutter", 
                   "exclaim", "call", "cry", "yell", "speak", "answer", "murmur"]
    
    for sent in sentences:
        # Check for quotation marks
        has_quotes = ('"' in sent) or ("'" in sent)
        
        if has_quotes:
            # Simple tokenization and tagging
            tokens = nltk.word_tokenize(sent)
            tagged = nltk.pos_tag(tokens)
            
            # Check for speech verbs
            has_speech_verb = False
            for token, tag in tagged:
                if tag.startswith('VB') and token.lower() in speech_verbs:
                    has_speech_verb = True
                    break
            
            if has_speech_verb or (sent.startswith('"') or sent.startswith("'")):
                dialogue_sentences.append(sent)
    
    end_time = time.time()
    return {
        "detected_dialogue": dialogue_sentences,
        "time_taken": end_time - start_time,
        "count": len(dialogue_sentences)
    }

def dialogue_detection_azure(text):
    """Detect dialogue using Azure (placeholder if Azure is not available)"""
    if not include_azure:
        return {
            "detected_dialogue": [],
            "time_taken": 0,
            "count": 0
        }
        
    # This would be implemented using the Azure client
    # Similar approach to NLTK for demonstration
    start_time = time.time()
    sentences = nltk.sent_tokenize(text)
    dialogue_sentences = [sent for sent in sentences if ('"' in sent or "'" in sent)]
    end_time = time.time()
    
    return {
        "detected_dialogue": dialogue_sentences,
        "time_taken": end_time - start_time,
        "count": len(dialogue_sentences)
    }

In [43]:
# Named Entity Recognition Functions (for character identification)

def ner_spacy(text):
    """Extract named entities using spaCy"""
    start_time = time.time()
    doc = nlp_lg(text)
    
    # Extract PERSON entities as potential characters
    entities = [(ent.text, ent.label_) for ent in doc.ents if ent.label_ == "PERSON"]
    
    # Count unique character names
    unique_characters = set(entity[0] for entity in entities)
    
    end_time = time.time()
    return {
        "entities": entities,
        "unique_characters": list(unique_characters),
        "time_taken": end_time - start_time,
        "count": len(unique_characters)
    }

def ner_nltk(text):
    """Extract named entities using NLTK"""
    start_time = time.time()
    
    entities = []
    sentences = nltk.sent_tokenize(text)
    
    for sent in sentences:
        tokens = nltk.word_tokenize(sent)
        tagged = nltk.pos_tag(tokens)
        chunks = nltk.ne_chunk(tagged)
        
        # Extract person entities
        for chunk in chunks:
            if hasattr(chunk, 'label') and chunk.label() == 'PERSON':
                name = ' '.join(c[0] for c in chunk)
                entities.append((name, 'PERSON'))
    
    # Count unique character names
    unique_characters = set(entity[0] for entity in entities)
    
    end_time = time.time()
    return {
        "entities": entities,
        "unique_characters": list(unique_characters),
        "time_taken": end_time - start_time,
        "count": len(unique_characters)
    }

def ner_azure(text):
    """Extract named entities using Azure (placeholder if Azure is not available)"""
    if not include_azure:
        return {
            "entities": [],
            "unique_characters": [],
            "time_taken": 0,
            "count": 0
        }
        
    # This would be implemented using the Azure client
    # Return empty results for now
    return {
        "entities": [],
        "unique_characters": [],
        "time_taken": 0,
        "count": 0
    }

In [44]:
# Sentiment Analysis Functions (for emotion detection)

def sentiment_spacy(text):
    """Analyze sentiment using spaCy with SpacyTextBlob"""
    start_time = time.time()
    
    # Using SpacyTextBlob for sentiment analysis
    doc = nlp_lg(text)
    
    # Get sentence-level sentiment
    sentences = [sent.text for sent in doc.sents]
    emotional_sentences = []
    
    for sent in sentences:
        sent_doc = nlp_lg(sent)
        # SpacyTextBlob provides polarity and subjectivity
        polarity = sent_doc._.blob.polarity
        subjectivity = sent_doc._.blob.subjectivity
        
        # Consider sentences with non-zero polarity and high subjectivity
        if abs(polarity) > 0.1 and subjectivity > 0.3:
            emotional_sentences.append((sent, polarity))
    
    # Sort by emotional intensity (absolute value of polarity)
    emotional_sentences.sort(key=lambda x: abs(x[1]), reverse=True)
    
    end_time = time.time()
    return {
        "emotional_sentences": emotional_sentences[:5],  # Top 5 most emotional
        "time_taken": end_time - start_time,
        "count": len(emotional_sentences)
    }

def sentiment_nltk(text):
    """Analyze sentiment using NLTK's VADER"""
    start_time = time.time()
    sid = SentimentIntensityAnalyzer()
    
    # Split text into paragraphs or sentences
    sentences = nltk.sent_tokenize(text)
    sentiment_scores = [sid.polarity_scores(sentence) for sentence in sentences]
    
    # Find most emotional sentences (highest absolute compound score)
    sentence_sentiments = [(sentences[i], scores["compound"]) for i, scores in enumerate(sentiment_scores)]
    emotional_sentences = sorted(sentence_sentiments, key=lambda x: abs(x[1]), reverse=True)[:5]
    
    end_time = time.time()
    return {
        "emotional_sentences": emotional_sentences,
        "time_taken": end_time - start_time,
        "count": len(sentence_sentiments)
    }

def sentiment_azure(text):
    """Analyze sentiment using Azure (placeholder if Azure is not available)"""
    if not include_azure:
        return {
            "emotional_sentences": [],
            "time_taken": 0,
            "count": 0
        }
        
    # This would be implemented using the Azure client
    # Return empty results for now
    return {
        "emotional_sentences": [],
        "time_taken": 0,
        "count": 0
    }

In [45]:
# Processing Speed Functions (for long-form content)

def speed_spacy(text):
    """Measure processing speed using spaCy"""
    start_time = time.time()
    doc = nlp_lg(text)
    
    # Perform some standard NLP operations
    _ = [token.text for token in doc]  # Tokenization
    _ = [token.pos_ for token in doc]  # POS tagging
    _ = [token.dep_ for token in doc]  # Dependency parsing
    _ = [ent for ent in doc.ents]      # NER
    
    end_time = time.time()
    processing_time = end_time - start_time
    chars_per_second = len(text) / processing_time if processing_time > 0 else 0
    
    return {
        "time_taken": processing_time,
        "chars_per_second": chars_per_second,
        "text_length": len(text)
    }

def speed_nltk(text):
    """Measure processing speed using NLTK"""
    start_time = time.time()
    
    # Perform standard NLP operations
    sentences = nltk.sent_tokenize(text)
    all_tokens = []
    for sent in sentences:
        tokens = nltk.word_tokenize(sent)
        all_tokens.extend(tokens)
    
    # Limit to first 10,000 tokens for POS tagging to avoid excessive time
    tagged = nltk.pos_tag(all_tokens[:10000])
    
    end_time = time.time()
    processing_time = end_time - start_time
    chars_per_second = len(text) / processing_time if processing_time > 0 else 0
    
    return {
        "time_taken": processing_time,
        "chars_per_second": chars_per_second,
        "text_length": len(text)
    }

def speed_azure(text):
    """Measure processing speed using Azure (placeholder if Azure is not available)"""
    if not include_azure:
        return {
            "time_taken": 0,
            "chars_per_second": 0,
            "text_length": len(text)
        }
        
    # This would be implemented using the Azure client
    # Return placeholder results
    return {
        "time_taken": 0,
        "chars_per_second": 0,
        "text_length": len(text)
    }

## 4. Main Benchmarking Functions

In [46]:
#Benchmarking Function
def benchmark_performance(text, task="all", libraries=["spacy", "nltk", "azure"]):
    """Benchmark NLP libraries on various tasks"""
    if "azure" in libraries and not include_azure:
        libraries.remove("azure")
        print("Azure benchmarking disabled. Set include_azure = True to enable.")
    
    results = {}
    
    # Define tests for each task and library
    tests = {
        "dialogue_detection": {
            "spacy": dialogue_detection_spacy,
            "nltk": dialogue_detection_nltk,
            "azure": dialogue_detection_azure
        },
        "ner": {
            "spacy": ner_spacy,
            "nltk": ner_nltk,
            "azure": ner_azure
        },
        "sentiment": {
            "spacy": sentiment_spacy,
            "nltk": sentiment_nltk,
            "azure": sentiment_azure
        },
        "processing_speed": {
            "spacy": speed_spacy,
            "nltk": speed_nltk,
            "azure": speed_azure
        }
    }
    
    # Run selected tests
    if task == "all":
        for t in tests:
            results[t] = {}
            for lib in libraries:
                if lib in tests[t]:
                    results[t][lib] = tests[t][lib](text)
    else:
        if task not in tests:
            print(f"Unknown task: {task}. Available tasks: {list(tests.keys())}")
            return {}
            
        results[task] = {}
        for lib in libraries:
            if lib in tests[task]:
                results[task][lib] = tests[task][lib](text)
    
    return results

In [47]:
def process_sample(sample_id, sample_text):
    """Function to process a single sample across all NLP tasks and libraries."""
    try:
        print(f"Running benchmarks on sample: {sample_id} (PID: {os.getpid()})")
        print(f"Sample length: {len(sample_text)} characters")

        # For very long texts, take a representative sample to avoid memory issues
        max_length = 100000  # Adjust based on your memory constraints
        if len(sample_text) > max_length:
            print(f"Sample too large, truncating to {max_length} characters for analysis")
            analysis_text = sample_text[:max_length]
        else:
            analysis_text = sample_text

        results = benchmark_performance(
            text=analysis_text,
            task="all",
            libraries=["spacy", "nltk"]
        )

        sample_results = []
        for task, lib_results in results.items():
            for lib, res in lib_results.items():
                result_row = {
                    "sample_id": sample_id,
                    "sample_length": len(sample_text),
                    "task": task,
                    "library": lib,
                    "time_taken": res["time_taken"]
                }

                # Add task-specific metrics
                if task == "dialogue_detection" and "count" in res:
                    result_row["count"] = res["count"]
                    result_row["sample_dialogues"] = res["detected_dialogue"][:3] if res["detected_dialogue"] else []
                elif task == "ner" and "count" in res:
                    result_row["unique_count"] = res["count"]
                    result_row["characters"] = res["unique_characters"][:10] if res["unique_characters"] else []
                elif task == "sentiment" and "count" in res:
                    result_row["emotional_count"] = res["count"]
                    result_row["emotional_sentences"] = res["emotional_sentences"][:3] if res["emotional_sentences"] else []
                elif task == "processing_speed" and "chars_per_second" in res:
                    result_row["chars_per_second"] = res["chars_per_second"]

                sample_results.append(result_row)
        
        # Force garbage collection to free memory
        gc.collect()
        
        return sample_results
    
    except Exception as e:
        print(f"Error processing sample {sample_id}: {e}")
        import traceback
        traceback.print_exc()
        
        # Force garbage collection even on error
        gc.collect()
        
        return []


In [ ]:
# Define a function to process text in chunks
def process_sample_in_chunks(sample_id, sample_text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Process a sample text in smaller, overlapping chunks to improve performance"""
    print(f"Processing {sample_id} in chunks (PID: {os.getpid()})")
    
    # For very small samples, just process the entire text
    if len(sample_text) <= chunk_size:
        return process_sample(sample_id, sample_text)
    
    # For larger texts, split into chunks with some overlap
    chunks = []
    sample_results = []
    
    # Split the text into overlapping chunks
    for i in range(0, len(sample_text), chunk_size - overlap):
        chunk_end = min(i + chunk_size, len(sample_text))
        chunks.append(sample_text[i:chunk_end])
    
    # Process each chunk
    for i, chunk in enumerate(chunks):
        chunk_id = f"{sample_id}#chunk{i+1}"
        print(f"  - Processing chunk {i+1}/{len(chunks)} ({len(chunk)} chars)")
        
        # For each chunk, we'll run NLP tasks
        results = benchmark_performance(
            text=chunk,
            task="all",
            libraries=["spacy", "nltk"]
        )
        
        # Collect results
        for task, lib_results in results.items():
            for lib, res in lib_results.items():
                result_row = {
                    "sample_id": sample_id,
                    "chunk_id": chunk_id,
                    "chunk_num": i+1,
                    "chunk_size": len(chunk),
                    "sample_length": len(sample_text),
                    "task": task,
                    "library": lib,
                    "time_taken": res["time_taken"]
                }
                
                # Add task-specific metrics
                if task == "dialogue_detection" and "count" in res:
                    result_row["count"] = res["count"]
                    result_row["sample_dialogues"] = res["detected_dialogue"][:3] if res["detected_dialogue"] else []
                elif task == "ner" and "count" in res:
                    result_row["unique_count"] = res["count"]
                    result_row["characters"] = res["unique_characters"][:10] if res["unique_characters"] else []
                elif task == "sentiment" and "count" in res:
                    result_row["emotional_count"] = res["count"]
                    result_row["emotional_sentences"] = res["emotional_sentences"][:3] if res["emotional_sentences"] else []
                elif task == "processing_speed" and "chars_per_second" in res:
                    result_row["chars_per_second"] = res["chars_per_second"]
                
                sample_results.append(result_row)
        
        # Force garbage collection after each chunk
        gc.collect()
    
    return sample_results

## 5. Main Execution Code

In [48]:
# Load sample texts
print("Loading sample texts...")
samples = load_samples()

if not samples:
    print("No sample texts found. Adding a small example for testing...")
    # Small sample text for testing if no other texts are available
    samples = {
        "test/sample.txt": """
        "Hello," said Alice to the White Rabbit. "Where are you going in such a hurry?"
        The rabbit looked at his pocket watch anxiously. "I'm late, I'm late, for a very important date!"
        Alice was curious. She followed the rabbit down the hole, feeling both excited and nervous.
        "This is quite unusual," she thought to herself.
        """
    }

print(f"Found {len(samples)} text samples for benchmarking")

Loading sample texts...
Loading text samples from directory: ../data/corpus
Loaded: classics/pride_and_prejudice-jane_austen.txt
Loaded: descriptive/the_rivers_of_great_britian-various_authors.txt
Loaded: dialogue_heavy/petrarchs_secret_or_the_souls_conflict_with_passion-francesco_petrarca.txt
Loaded: emotional/the_sorrows_of_young_werther-johann_wolfgang_von_goethe.txt
Loaded: modern/a_modern_utopia-hg_wells.txt
Found 5 text samples for benchmarking


In [ ]:
# Start with memory cleanup to ensure a fresh run
print("Starting fresh run - cleaning up memory...")
clear_memory()

# Set optimal parameters for your hardware
# With 64GB RAM and i7-11800H (8 cores/16 threads), we can be more aggressive
CHUNK_SIZE = 75000    # Larger chunks since you have plenty of RAM
OVERLAP = 1000        # Larger overlap for better context preservation
MAX_WORKERS = 6       # More workers, but still leaving headroom for system

# Run benchmarks on all samples using ProcessPoolExecutor with tqdm progress bar
all_results = []

# Define a worker initialization function
def init_worker():
    """Initialize each worker process"""
    print(f"Initializing worker process {os.getpid()}")
    
    # Load models in each worker process to avoid sharing issues
    global nlp_sm, nlp_lg
    try:
        import spacy
        nlp_sm = spacy.load("en_core_web_sm")
        print(f"Worker {os.getpid()} loaded spaCy small model")
        
        nlp_lg = spacy.load("en_core_web_lg")
        if 'spacytextblob' not in nlp_lg.pipe_names:
            from spacytextblob.spacytextblob import SpacyTextBlob
            nlp_lg.add_pipe("spacytextblob")
        print(f"Worker {os.getpid()} loaded spaCy large model with SpacyTextBlob")
    except Exception as e:
        print(f"Worker {os.getpid()} error loading spaCy models: {e}")

# Use ProcessPoolExecutor for true parallelism
from concurrent.futures import ProcessPoolExecutor

print(f"Using {MAX_WORKERS} parallel processes for benchmarking with chunk size of {CHUNK_SIZE} chars")
sample_items = list(samples.items())

# Process samples in parallel using ProcessPoolExecutor
with ProcessPoolExecutor(max_workers=MAX_WORKERS, initializer=init_worker) as executor:
    futures = {executor.submit(process_sample_in_chunks, sample_id, sample_text): sample_id 
              for sample_id, sample_text in sample_items}
    
    # Create a progress bar with tqdm
    for future in tqdm(as_completed(futures), total=len(futures), 
                      desc="Processing samples", unit="sample"):
        sample_id = futures[future]
        try:
            sample_result = future.result()
            if sample_result:  # Check if result is not empty
                all_results.extend(sample_result)
                print(f"✓ Completed: {sample_id} ({len(sample_result)} results)")
            else:
                print(f"✗ Failed: {sample_id} (no results)")
        except Exception as e:
            print(f"✗ Error: {sample_id} - {e}")
            
# Final memory cleanup
print("Processing complete - cleaning up memory...")
clear_memory()

Using 4 parallel threads for benchmarking
Running benchmarks on sample: classics/pride_and_prejudice-jane_austen.txt (PID: 173352)
Sample length: 748126 characters
Sample too large, truncating to 100000 characters for analysis
Running benchmarks on sample: descriptive/the_rivers_of_great_britian-various_authors.txt (PID: 173352)
Sample length: 924103 characters
Sample too large, truncating to 100000 characters for analysis
Running benchmarks on sample: dialogue_heavy/petrarchs_secret_or_the_souls_conflict_with_passion-francesco_petrarca.txt (PID: 173352)
Sample length: 296703 characters
Sample too large, truncating to 100000 characters for analysis
Running benchmarks on sample: emotional/the_sorrows_of_young_werther-johann_wolfgang_von_goethe.txt (PID: 173352)
Sample length: 254633 characters
Sample too large, truncating to 100000 characters for analysis


Processing samples:   0%|          | 0/5 [00:00<?, ?sample/s]

## 6. Results Analysis and Visualization

In [ ]:
# Convert results to DataFrame for analysis
results_df = pd.DataFrame(all_results)
print("\n===== BENCHMARK RESULTS SUMMARY =====\n")

# Check if we have results
if results_df.empty:
    print("No benchmark results available. Please check for errors in the processing.")
else:
    summary = results_df.groupby(['task', 'library']).agg({
        'time_taken': ['mean', 'min', 'max', 'std'],
        'sample_length': ['mean', 'count']
    })
    print(summary)

    # Calculate average performance by library and task
    print("\n===== LIBRARY PERFORMANCE COMPARISON =====\n")
    library_task_performance = results_df.pivot_table(
        index='library', 
        columns='task', 
        values='time_taken', 
        aggfunc='mean'
    )
    print("Average processing time (seconds) by library and task:")
    print(library_task_performance)

    library_performance = results_df.groupby('library')['time_taken'].mean()
    print("\nOverall average processing time (seconds):")
    print(library_performance)

    # Save detailed results
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path("../results")
    results_dir.mkdir(parents=True, exist_ok=True)  # Create directory if it doesn't exist

    # Save results as JSON
    result_file = results_dir / f"benchmark_results_{timestamp}.json"
    with open(result_file, "w") as f:
        json.dump({
            "sample_count": len(samples),
            "timestamp": str(datetime.datetime.now()),
            "summary": {
                "library_performance": {lib: time for lib, time in library_performance.items()},
                "task_performance": library_task_performance.to_dict()
            },
            "results": all_results
        }, f, indent=2, default=str)

    print(f"\nDetailed results saved to {result_file}")

    # Create CSV for easier analysis
    csv_file = results_dir / f"benchmark_results_{timestamp}.csv"
    results_df.to_csv(csv_file, index=False)
    print(f"CSV results saved to {csv_file}")

In [ ]:
# Create visualizations
try:
    if not results_df.empty:
        # Set style
        sns.set(style="whitegrid")
        
        # Create a figure with multiple subplots
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        # 1. Processing time by task and library (bar chart)
        time_by_task = results_df.pivot_table(
            index='task', 
            columns='library', 
            values='time_taken', 
            aggfunc='mean'
        )
        time_by_task.plot(kind='bar', ax=axes[0, 0], title='Average Processing Time by Task')
        axes[0, 0].set_ylabel('Time (seconds)')
        axes[0, 0].set_xlabel('Task')
        
        # 2. Processing speed comparison (bar chart)
        speed_data = results_df[results_df['task'] == 'processing_speed']
        if not speed_data.empty:
            sns.barplot(x='library', y='chars_per_second', data=speed_data, ax=axes[0, 1])
            axes[0, 1].set_title('Characters Processed Per Second')
            axes[0, 1].set_ylabel('Chars/second')
            axes[0, 1].set_xlabel('Library')
        
        # 3. NER performance (scatter plot)
        ner_data = results_df[results_df['task'] == 'ner']
        if not ner_data.empty:
            scatter = sns.scatterplot(
                x='sample_length', 
                y='time_taken', 
                hue='library', 
                size='unique_count', 
                sizes=(20, 200),
                data=ner_data, 
                ax=axes[1, 0]
            )
            axes[1, 0].set_title('NER Performance by Sample Size')
            axes[1, 0].set_xlabel('Sample Length (chars)')
            axes[1, 0].set_ylabel('Processing Time (seconds)')
            # Add legend title if needed
            if scatter.get_legend() is not None:
                handles, labels = scatter.get_legend_handles_labels()
                if len(handles) > 2:  # Make sure we have size-related handles
                    axes[1, 0].legend(handles=handles[2:], labels=labels[2:], title="Character Count")
        
        # 4. Time vs. text length for all tasks (line plot)
        sns.lineplot(
            x='sample_length', 
            y='time_taken', 
            hue='library',
            style='task',
            markers=True,
            data=results_df, 
            ax=axes[1, 1]
        )
        axes[1, 1].set_title('Processing Time vs. Text Length')
        axes[1, 1].set_xlabel('Sample Length (chars)')
        axes[1, 1].set_ylabel('Time (seconds)')
        
        plt.tight_layout()
        
        # Save the figure
        plot_file = results_dir / f"benchmark_plots_{timestamp}.png"
        plt.savefig(plot_file, dpi=300)
        print(f"Visualization saved to {plot_file}")
        
        # Additional visualizations
        # 1. Box plot of processing times
        plt.figure(figsize=(12, 6))
        sns.boxplot(x='task', y='time_taken', hue='library', data=results_df)
        plt.title('Distribution of Processing Times by Task and Library')
        plt.ylabel('Time (seconds)')
        plt.tight_layout()
        box_plot_file = results_dir / f"benchmark_boxplot_{timestamp}.png"
        plt.savefig(box_plot_file, dpi=300)
        print(f"Box plot saved to {box_plot_file}")
        
        # 2. Heatmap of time correlation between tasks
        plt.figure(figsize=(10, 8))
        # Pivot and calculate correlation
        task_pivot = results_df.pivot_table(
            index=['sample_id', 'library'], 
            columns='task', 
            values='time_taken'
        )
        corr = task_pivot.corr()
        sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
        plt.title('Correlation of Processing Times Between Tasks')
        plt.tight_layout()
        corr_plot_file = results_dir / f"benchmark_correlation_{timestamp}.png"
        plt.savefig(corr_plot_file, dpi=300)
        print(f"Correlation plot saved to {corr_plot_file}")
        
        # 3. Direct comparison of libraries
        plt.figure(figsize=(10, 6))
        comp_data = results_df.pivot_table(
            index='sample_id',
            columns='library',
            values='time_taken',
            aggfunc='sum'
        )
        # Add a 45-degree line to show equal performance
        max_val = max(comp_data.max().max(), 1)
        plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='Equal performance')
        
        # Plot scatter plot of spaCy vs NLTK performance
        if 'spacy' in comp_data.columns and 'nltk' in comp_data.columns:
            plt.scatter(comp_data['spacy'], comp_data['nltk'], alpha=0.7)
            plt.xlabel('spaCy Processing Time (seconds)')
            plt.ylabel('NLTK Processing Time (seconds)')
            plt.title('Direct Comparison: spaCy vs NLTK Performance')
            # Add text labels for points that are far from the equal performance line
            for idx, row in comp_data.iterrows():
                if abs(row['spacy'] - row['nltk']) > max(row['spacy'], row['nltk']) * 0.5:
                    plt.text(row['spacy'], row['nltk'], idx.split('/')[-1], fontsize=8)
            plt.tight_layout()
            comp_plot_file = results_dir / f"library_comparison_{timestamp}.png"
            plt.savefig(comp_plot_file, dpi=300)
            print(f"Library comparison plot saved to {comp_plot_file}")
        
        # Display all plots
        plt.show()
        
    else:
        print("No data available for visualization. Check for errors in the processing.")
except Exception as e:
    print(f"Could not create visualizations: {e}")
    import traceback
    traceback.print_exc()

print("\nBenchmark completed!")

In [ ]:
# Use ThreadPoolExecutor instead of ProcessPoolExecutor for spaCy compatibility
from concurrent.futures import ThreadPoolExecutor

# For CPU-bound tasks with multiple large models loaded, 
# limiting concurrent threads can prevent memory issues
max_workers = min(4, os.cpu_count(), len(samples))
print(f"Using {max_workers} parallel threads for benchmarking")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(process_sample, sample_id, sample_text): sample_id 
               for sample_id, sample_text in samples.items()}
    
    for future in as_completed(futures):
        sample_id = futures[future]
        try:
            sample_result = future.result()
            if sample_result:  # Check if result is not empty
                all_results.extend(sample_result)
        except Exception as e:
            print(f"Error processing sample {sample_id}: {e}")